# PlateBendingElement Finite Difference Validation

In [ ]:
import sys; sys.path.append('..')
import MeshFEM, elastic_sheet, tensors
import numpy as np
np.set_printoptions(edgeitems=12,linewidth=280)

In [ ]:
import fd_validation

In [ ]:
class FDWrap:
    def __init__(self):
        self.pbe = elastic_sheet.PlateBendingElement(1)
        self.X = np.random.normal(size=(3,3))
        self.x = self.X + 0.1 * np.random.normal(size=(3,3))
        self.gamma = 0.01 * np.random.normal(size=3)
        self.C = tensors.ElasticityTensor2D(2000, 0.3)
        
    def numVars(self): return 12
    def getVars(self): return np.concatenate((self.x.ravel(), self.gamma))
    def setVars(self, v):
        self.x = v[0:9].reshape(3, 3)
        self.gamma = v[9:]
    def energy(self):
        return self.pbe.energy(self.C, self.X, self.x, self.gamma)
    def gradient(self):
        return self.pbe.gradient(self.C, self.X, self.x, self.gamma)
    def hessian(self):
        return self.pbe.hessian(self.C, self.X, self.x, self.gamma)

In [ ]:
fdwrap = FDWrap()

In [ ]:
fd_validation.gradConvergencePlot(fdwrap)

In [ ]:
fd_validation.hessConvergencePlot(fdwrap)

In [ ]:
fd_validation.hessConvergencePlot(fdwrap, perturb=fd_validation.basisDirection(fdwrap, 10))

In [ ]:
fdwrap.hessian()

In [ ]:
fd_validation.fdHessian(fdwrap, 1e-6)

# Dihedral Angle Validation Finite Difference Validation

In [ ]:
import scipy.spatial.transform

In [ ]:
def randomTriFlap(theta = None):
    if theta is None: theta = np.pi * np.random.normal()
    h2 = np.abs(np.random.normal())
    V = np.array([[0, 0, -np.abs(np.random.normal())], [0, 0, np.abs(np.random.normal())], [np.abs(np.random.normal()), 0, 0], [-h2 * np.cos(theta), (-h2) * np.sin(theta), 0]])
    V = V @ scipy.spatial.transform.Rotation.random().as_matrix().T
    return V

In [ ]:
randomTriFlap()

In [ ]:
class FDWrap:
    def __init__(self):
        self.da = elastic_sheet.DihedralAngle()
        self.setVars(randomTriFlap())
    def numVars(self): return self.pts.size
    def getVars(self): return self.pts.ravel()
    def setVars(self, v):
        self.pts = v.reshape((4,3))
        self.da.configure(self.pts)
    def energy(self): return self.da.value()
    def gradient(self): return self.da.gradient()
    def hessian(self): return self.da.hessian()

In [ ]:
fdwrap = FDWrap()

In [ ]:
fdwrap.energy()

In [ ]:
fd_validation.fdGrad(fdwrap, 1e-6) / fdwrap.gradient()

In [ ]:
fd_validation.gradConvergencePlot(fdwrap)

In [ ]:
fd_validation.hessConvergencePlot(fdwrap, perturb=fd_validation.basisDirection(fdwrap, 10))

In [ ]:
fd_validation.fdHessian(fdwrap, 1e-6)

In [ ]:
fdwrap.hessian() / fd_validation.fdHessian(fdwrap, 1e-6)